#### Monthly Order Summary
for each of the custmoer, produce the following summary per month

1. total orders
2. total items bought
3. total amount spent

In [0]:
df_orders = spark.table('gizmobox_catalog_subbu.silver.py_orders')
display(df_orders)

In [0]:
from pyspark.sql import functions as F
df_order_summary = (
                df_orders
                # type 1
                # .select('order_id','order_date','total_amount','order_status','payment_method','customer_id','transaction_timestamp','item_id','name','category','price','quantity','brand','color',
                #        F.date_format('transaction_timestamp','yyyy-MM').alias('order_month'))
                #  .groupBy('order_month','customer_id')


                # type 2
                .withColumn("order_month", F.date_format('transaction_timestamp','yyyy-MM'))        
                .groupBy('order_month','customer_id')
                
                #type 3
                # .groupBy(F.date_format('transaction_timestamp','yyyy-MM').alias('order_month'), 'customer_id')
               
                .agg(
                    F.count('order_id').alias('total_orders'),
                    F.sum('quantity').alias('total_items_bought'),
                    F.sum(F.col('price') * F.col('quantity')).alias('total_amount')
                )
)
display(df_order_summary)

In [0]:
df_order_summary.writeTo('gizmobox_catalog_subbu.gold.py_order_summary_monthly').createOrReplace()

In [0]:
%sql
select * from gizmobox_catalog_subbu.gold.py_order_summary_monthly